# Stage 3.5 - Visualizing Pixel Importance with Heatmaps (Grad Cam)

**Input:**
* (pre-)trained models (e.g. checkpoints)
* 5 representative pre-chosen example images (easy/hard)

Run tests with: 
    * different frozen backbone models - metric: self-similarity of embedding
    * background: normal | blurred | neutral

**Output:**
* Heatmap of each image per model evaluated on self-similarity of embedding

**Resources:**
* [pytorch GradCam tutorial](https://gitlab.db.in.tum.de/anamlacatusu/reid-or/-/blob/main/pytorch_grad_cam/tutorials/vision_transformers.md)
* [GradCam pyPpi](https://pypi.org/project/grad-cam/1.3.5/)
* [FiftyOne Tutorial for GradCam](https://voxel51.com/blog/exploring-gradcam-and-more-with-fiftyone)

In [ ]:
import os

os.environ["PYTHONHASHSEED"] = "51"

In [7]:
import sys
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

%cd "/content/drive/MyDrive/Colab Notebooks/Applied Computer Vision/Project_Scratchpad Ideas/"

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

sys.path.insert(0, str(PROJECT_ROOT / "src"))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/Colab Notebooks/Applied Computer Vision/Project_Scratchpad Ideas


In [4]:
# Install dependencies
%pip install --no-cache-dir -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 222.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 288.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 278.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.6/112.6 kB 377.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 332.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.4/112.4 kB 363.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.8/74.8 kB 335.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 93.9 MB/s eta 0:00:00
   ━━━━━━━━━━

In [ ]:
from PIL import Image
import shutil

In [9]:
from config import DEVICE, SEED, NUM_WORKERS
from models import load_dinov2, load_mega_descriptor
from evaluation import FacebookDinoWrapper, MegaDescriptorWrapper, compute_grad_cam
from preprocessing import get_gray_background_from_rgba
from utils import get_timestamp, ensure_dir

In [ ]:
# set paths
DRIVE_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/Applied Computer Vision/Project_Scratchpad Ideas")
DATA_ROOT = DRIVE_ROOT / "datasets"
DATA_DIR = DATA_ROOT / "out"

OUT_ROOT = Path("/content/out")
DRIVE_OUT_DIR = DRIVE_ROOT / "results/gradCam/"

ensure_dir([DATA_DIR, OUT_ROOT, DRIVE_OUT_DIR])

In [10]:
# Define Input Images
image_paths = [
    DATA_DIR / "foreground_rgba/Aju__right__fg.png",
]

In [11]:
# Load Model
dinov2, dinov2_transform = load_dinov2(dinoname="dinov2_vitb14")

Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to /root/.cache/torch/hub/main.zip


/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vitb14/dinov2_vitb14_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dinov2_vitb14_pretrain.pth


100%|██████████| 330M/330M [00:02<00:00, 126MB/s]


In [12]:
# Load Model
mega_descriptor, mega_descriptor_transform = load_mega_descriptor()

config.json:   0%|          | 0.00/609 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.94G [00:00<?, ?B/s]

In [13]:
models = [
    ( "DINOv2_Base", dinov2, dinov2_transform, ),
    ( "MegaDescriptor_L", mega_descriptor, mega_descriptor_transform, )
]

In [ ]:
for name, model, transform in models:
    imgs = []
    for img_path in image_paths:
        img_pil = get_gray_background_from_rgba(img_path)
        imgs.append(img_pil)

    heatmaps = compute_grad_cam(
        model_config=(name, model, transform),
        imgs=imgs
    )

    for i, heatmap in enumerate(heatmaps):
        timestamp = get_timestamp()
        save_path = OUT_ROOT / f"heatmap_{i+1}_{name}_{timestamp}.png"
        Image.fromarray(heatmap).save(save_path)

    print(f"Success! Saved {i} heatmaps of {name}")

Creating Heatmaps...


100%|██████████| 1/1 [00:00<00:00,  2.86it/s]


Success! Saved 0 heatmaps of DINOv2_Base
Creating Heatmaps...


100%|██████████| 1/1 [00:00<00:00,  5.24it/s]


Success! Saved 0 heatmaps of MegaDescriptor_L


In [ ]:
shutil.copytree(OUT_ROOT, DRIVE_OUT_DIR, dirs_exist_ok=True)

PosixPath('/content/drive/MyDrive/Colab Notebooks/Applied Computer Vision/Project_Scratchpad Ideas/results/gradCam')